In [ ]:
%sql
-- Databricks SQL script: Enriches patient_therapy_shipment with derived adherence, gap, age, and discontinuation metrics
-- Purpose: Calculate and display 12 derived columns and 14 direct columns for patient therapy shipment enrichment
-- Author: Giang Nguyen
-- Date: 2025-10-13
-- Description: Reads purgo_playground.patient_therapy_shipment, applies business logic for derived metrics (adherence, gaps, age, discontinuation), and displays enriched results for analytics teams

USE CATALOG purgo_databricks;

-- CTE: Enrich patient_therapy_shipment with derived metrics per business logic

-- Final SELECT: Display enriched result for downstream analytics, filter to one row per patient_id, treatment_id, ship_date

WITH enriched_shipment AS (
  SELECT
    -- Source columns, explicit type casting for schema consistency
    CAST(product AS STRING) AS product,
    CAST(ship_date AS DATE) AS ship_date,
    COALESCE(CAST(days_supply AS STRING), "") AS days_supply,
    COALESCE(CAST(qty AS STRING), "") AS qty,
    CAST(treatment_id AS STRING) AS treatment_id,
    CAST(dob AS DATE) AS dob,
    CAST(first_ship_date AS DATE) AS first_ship_date,
    CAST(refill_status AS STRING) AS refill_status,
    CAST(patient_id AS STRING) AS patient_id,
    CAST(ship_type AS STRING) AS ship_type,
    CAST(shipment_arrived_status AS STRING) AS shipment_arrived_status,
    CAST(delivery_ontime AS STRING) AS delivery_ontime,
    -- Derived columns
    -- shipment_expiry: Add days_supply (or fallback) to ship_date
    DATE_ADD(
      ship_date,
      COALESCE(
        CAST(days_supply AS INT),
        CASE
          WHEN qty IS NOT NULL AND qty != ""
            THEN CAST(qty AS INT) / 3 * 7
          ELSE NULL
        END
      )
    ) AS shipment_expiry,
    -- discontinuation_date: shipment_expiry + 91 days
    DATE_ADD(
      DATE_ADD(
        ship_date,
        COALESCE(
          CAST(days_supply AS INT),
          CASE
            WHEN qty IS NOT NULL AND qty != ""
              THEN CAST(qty AS INT) / 3 * 7
            ELSE NULL
          END
        )
      ),
      91
    ) AS discontinuation_date,
    -- days_until_next_ship: shipment_expiry - calctime + 1
    DATEDIFF(
      DATE_ADD(
        ship_date,
        COALESCE(
          CAST(days_supply AS INT),
          CASE
            WHEN qty IS NOT NULL AND qty != ""
              THEN CAST(qty AS INT) / 3 * 7
            ELSE NULL
          END
        )
      ),
      calctime
    ) + 1 AS days_until_next_ship,
    -- days_since_last_fill: calctime - ship_date + 1
    DATEDIFF(calctime, ship_date) + 1 AS days_since_last_fill,
    -- expected_refill_date: calctime + days_until_next_ship
    DATE_ADD(
      calctime,
      DATEDIFF(
        DATE_ADD(
          ship_date,
          COALESCE(
            CAST(days_supply AS INT),
            CASE
              WHEN qty IS NOT NULL AND qty != ""
                THEN CAST(qty AS INT) / 3 * 7
              ELSE NULL
            END
          )
        ),
        calctime
      ) + 1
    ) AS expected_refill_date,
    -- prior_ship: previous ship_date for same treatment_id
    LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date) AS prior_ship,
    -- days_between: ship_date - prior_ship
    CASE
      WHEN LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date) IS NOT NULL
        THEN DATEDIFF(
          ship_date,
          LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date)
        )
      ELSE NULL
    END AS days_between,
    -- days_since_supply_out: calctime - shipment_expiry if >= 0
    CASE
      WHEN DATEDIFF(
        calctime,
        DATE_ADD(
          ship_date,
          COALESCE(
            CAST(days_supply AS INT),
            CASE
              WHEN qty IS NOT NULL AND qty != ""
                THEN CAST(qty AS INT) / 3 * 7
              ELSE NULL
            END
          )
        )
      ) >= 0
        THEN DATEDIFF(
          calctime,
          DATE_ADD(
            ship_date,
            COALESCE(
              CAST(days_supply AS INT),
              CASE
                WHEN qty IS NOT NULL AND qty != ""
                  THEN CAST(qty AS INT) / 3 * 7
                ELSE NULL
              END
            )
          )
        )
      ELSE NULL
    END AS days_since_supply_out,
    -- age: floor((calctime - dob)/365.25)
    CASE
      WHEN dob IS NOT NULL AND calctime IS NOT NULL
        THEN FLOOR(DATEDIFF(calctime, dob) / 365.25)
      ELSE NULL
    END AS age,
    -- age_at_first_ship: round((first_ship_date - dob)/365.0, 0)
    CASE
      WHEN dob IS NOT NULL AND first_ship_date IS NOT NULL
        THEN ROUND(DATEDIFF(first_ship_date, dob) / 365.0, 0)
      ELSE NULL
    END AS age_at_first_ship,
    -- latest_therapy_ships: count of commercial shipments per patient_id, treatment_id
    COUNT(
      CASE WHEN ship_type = "commercial" THEN ship_date END
    ) OVER(PARTITION BY patient_id, treatment_id) AS latest_therapy_ships,
    -- discontinuation_type: map refill_status to type
    CASE
      WHEN refill_status = "DC - Standard" THEN "STANDARD"
      WHEN refill_status = "DC-PERMANENT" THEN "PERMANENT"
      ELSE NULL
    END AS discontinuation_type,
    calctime
  FROM purgo_playground.patient_therapy_shipment
  -- Data quality validation: Exclude rows with missing required fields or invalid dates
  WHERE
    product IS NOT NULL AND product != ""
    AND ship_date IS NOT NULL
    AND treatment_id IS NOT NULL AND treatment_id != ""
    AND patient_id IS NOT NULL AND patient_id != ""
    AND dob IS NOT NULL
    AND first_ship_date IS NOT NULL
    AND qty IS NOT NULL AND qty != ""
    AND (
      TO_DATE(ship_date) IS NOT NULL
      AND TO_DATE(dob) IS NOT NULL
      AND TO_DATE(first_ship_date) IS NOT NULL
    )
)

SELECT
  product,
  ship_date,
  days_supply,
  qty,
  treatment_id,
  dob,
  first_ship_date,
  refill_status,
  patient_id,
  ship_type,
  shipment_arrived_status,
  delivery_ontime,
  shipment_expiry,
  discontinuation_date,
  days_until_next_ship,
  days_since_last_fill,
  expected_refill_date,
  prior_ship,
  days_between,
  days_since_supply_out,
  age,
  age_at_first_ship,
  latest_therapy_ships,
  discontinuation_type
FROM (

WITH enriched_shipment AS (
  SELECT
    -- Source columns, explicit type casting for schema consistency
    CAST(product AS STRING) AS product,
    CAST(ship_date AS DATE) AS ship_date,
    COALESCE(CAST(days_supply AS STRING), "") AS days_supply,
    COALESCE(CAST(qty AS STRING), "") AS qty,
    CAST(treatment_id AS STRING) AS treatment_id,
    CAST(dob AS DATE) AS dob,
    CAST(first_ship_date AS DATE) AS first_ship_date,
    CAST(refill_status AS STRING) AS refill_status,
    CAST(patient_id AS STRING) AS patient_id,
    CAST(ship_type AS STRING) AS ship_type,
    CAST(shipment_arrived_status AS STRING) AS shipment_arrived_status,
    CAST(delivery_ontime AS STRING) AS delivery_ontime,
    -- Derived columns
    -- shipment_expiry: Add days_supply (or fallback) to ship_date
    DATE_ADD(
      ship_date,
      COALESCE(
        CAST(days_supply AS INT),
        CASE
          WHEN qty IS NOT NULL AND qty != ""
            THEN CAST(qty AS INT) / 3 * 7
          ELSE NULL
        END
      )
    ) AS shipment_expiry,
    -- discontinuation_date: shipment_expiry + 91 days
    DATE_ADD(
      DATE_ADD(
        ship_date,
        COALESCE(
          CAST(days_supply AS INT),
          CASE
            WHEN qty IS NOT NULL AND qty != ""
              THEN CAST(qty AS INT) / 3 * 7
            ELSE NULL
          END
        )
      ),
      91
    ) AS discontinuation_date,
    -- days_until_next_ship: shipment_expiry - calctime + 1
    DATEDIFF(
      DATE_ADD(
        ship_date,
        COALESCE(
          CAST(days_supply AS INT),
          CASE
            WHEN qty IS NOT NULL AND qty != ""
              THEN CAST(qty AS INT) / 3 * 7
            ELSE NULL
          END
        )
      ),
      calctime
    ) + 1 AS days_until_next_ship,
    -- days_since_last_fill: calctime - ship_date + 1
    DATEDIFF(calctime, ship_date) + 1 AS days_since_last_fill,
    -- expected_refill_date: calctime + days_until_next_ship
    DATE_ADD(
      calctime,
      DATEDIFF(
        DATE_ADD(
          ship_date,
          COALESCE(
            CAST(days_supply AS INT),
            CASE
              WHEN qty IS NOT NULL AND qty != ""
                THEN CAST(qty AS INT) / 3 * 7
              ELSE NULL
            END
          )
        ),
        calctime
      ) + 1
    ) AS expected_refill_date,
    -- prior_ship: previous ship_date for same treatment_id
    LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date) AS prior_ship,
    -- days_between: ship_date - prior_ship
    CASE
      WHEN LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date) IS NOT NULL
        THEN DATEDIFF(
          ship_date,
          LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date)
        )
      ELSE NULL
    END AS days_between,
    -- days_since_supply_out: calctime - shipment_expiry if >= 0
    CASE
      WHEN DATEDIFF(
        calctime,
        DATE_ADD(
          ship_date,
          COALESCE(
            CAST(days_supply AS INT),
            CASE
              WHEN qty IS NOT NULL AND qty != ""
                THEN CAST(qty AS INT) / 3 * 7
              ELSE NULL
            END
          )
        )
      ) >= 0
        THEN DATEDIFF(
          calctime,
          DATE_ADD(
            ship_date,
            COALESCE(
              CAST(days_supply AS INT),
              CASE
                WHEN qty IS NOT NULL AND qty != ""
                  THEN CAST(qty AS INT) / 3 * 7
                ELSE NULL
              END
            )
          )
        )
      ELSE NULL
    END AS days_since_supply_out,
    -- age: floor((calctime - dob)/365.25)
    CASE
      WHEN dob IS NOT NULL AND calctime IS NOT NULL
        THEN FLOOR(DATEDIFF(calctime, dob) / 365.25)
      ELSE NULL
    END AS age,
    -- age_at_first_ship: round((first_ship_date - dob)/365.0, 0)
    CASE
      WHEN dob IS NOT NULL AND first_ship_date IS NOT NULL
        THEN ROUND(DATEDIFF(first_ship_date, dob) / 365.0, 0)
      ELSE NULL
    END AS age_at_first_ship,
    -- latest_therapy_ships: count of commercial shipments per patient_id, treatment_id
    COUNT(
      CASE WHEN ship_type = "commercial" THEN ship_date END
    ) OVER(PARTITION BY patient_id, treatment_id) AS latest_therapy_ships,
    -- discontinuation_type: map refill_status to type
    CASE
      WHEN refill_status = "DC - Standard" THEN "STANDARD"
      WHEN refill_status = "DC-PERMANENT" THEN "PERMANENT"
      ELSE NULL
    END AS discontinuation_type,
    calctime
  FROM purgo_playground.patient_therapy_shipment
  -- Data quality validation: Exclude rows with missing required fields or invalid dates
  WHERE
    product IS NOT NULL AND product != ""
    AND ship_date IS NOT NULL
    AND treatment_id IS NOT NULL AND treatment_id != ""
    AND patient_id IS NOT NULL AND patient_id != ""
    AND dob IS NOT NULL
    AND first_ship_date IS NOT NULL
    AND qty IS NOT NULL AND qty != ""
    AND (
      TO_DATE(ship_date) IS NOT NULL
      AND TO_DATE(dob) IS NOT NULL
      AND TO_DATE(first_ship_date) IS NOT NULL
    )
)

SELECT
    product,
    ship_date,
    days_supply,
    qty,
    treatment_id,
    dob,
    first_ship_date,
    refill_status,
    patient_id,
    ship_type,
    shipment_arrived_status,
    delivery_ontime,
    shipment_expiry,
    discontinuation_date,
    days_until_next_ship,
    days_since_last_fill,
    expected_refill_date,
    prior_ship,
    days_between,
    days_since_supply_out,
    age,
    age_at_first_ship,
    latest_therapy_ships,
    discontinuation_type,
    ROW_NUMBER() OVER(PARTITION BY patient_id, treatment_id, ship_date ORDER BY calctime) AS rn
  FROM enriched_shipment
) WHERE rn = 1
-- End of script
